<a href="https://colab.research.google.com/github/CarlKo-DLSU/PowerpuffCarl-MP1/blob/main/MP_Problem3_PowerpuffCarl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#MP1 - Problem 3 | Powerpuff Carl

This notebook solves the following quantum problem:

Find a real angle θ for which the single-qubit rotation RX(θ)
implements the Pauli-X gate, up to a global phase.

Mathematically:

    RX(θ) = e^{-i θ X / 2}
and we require:

    RX(θ) = e^{i φ} X

Because global phase e^{i φ} is physically irrelevant, we want RX(θ)
to act exactly like X on quantum states.

We know analytically that:

    cos(θ/2) = 0  -->  θ = π + 2πk  (for integer k)

Nevertheless, this notebook uses:
  - numeric grid search,
  - local refinement,
  - analytic correction,
  - and Qiskit verification,

to demonstrate how one would *algorithmically* find such an angle.
This is useful in gate-synthesis, calibration, and demonstration contexts.

# Install required quantum, math, and formatting libraries

This block installs the minimal set of Python packages needed:

qiskit: Used for two purposes:
1. Building a 1-qubit RX(θ) circuit to obtain the actual unitary
matrix that Qiskit uses internally.
2. Comparing the Qiskit-generated RX matrix to our numpy formula.

numpy:
    Needed for matrix algebra, trigonometric functions, and vector norms.

pylatexenc:
    Used only to render LaTeX labels (RX(θ), etc.) more cleanly in text output.

In [ ]:
!pip install qiskit
!pip install numpy
!pip install pylatexenc

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=76373c2d1c861a8aa571c0f6dd1974f70ad1a651c95bc54a840852eec507d1d5
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


# Import core libraries

This block imports the tools used throughout the notebook:

math, numpy:
    For scalar and matrix-level computations — these implement the analytic
    RX(θ) formula and enable the global-phase-invariant distance metric.

pylatexenc:
    Converts LaTeX strings like "RX(\theta)" into plain UTF-8 text so the
    printed output in the notebook looks clean.

Qiskit QuantumCircuit + Operator:
    These are optional but extremely useful:
    
- QuantumCircuit.rx(θ) constructs the real RX gate used by Qiskit.
- Operator(qc).data extracts the exact 2×2 matrix representation.

  Using Qiskit's version ensures we are validating a real-world
  implementation of RX, not just our derived analytic formula.

In [ ]:
import math
import numpy as np
from pylatexenc.latex2text import LatexNodes2Text

# optional: Qiskit Operator for verification (used to demonstrate dependency usage)
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator

# Pauli-X Definition

Here we explicitly define the Pauli-X gate:

    X = [[0, 1], [1, 0]]

We store it as a complex numpy matrix. This allows:
  - Matrix multiplication with RX(θ)
  - Comparison via global-phase-invariant distance
  - Evaluating its action on |0⟩, |1⟩

This is the "target" unitary our RX(θ) should match up to global phase.

In [ ]:
X = np.array([[0, 1], [1, 0]], dtype=complex)

# Analytic RX Matrix via Numpy

Implement the exact formula:

    RX(θ) = cos(θ/2) I – i sin(θ/2) X

Here we construct the 2×2 matrix using numpy primitives. This serves two roles:
  1. Enables extremely fast evaluations during the grid + refine search.
  2. Lets us compare a mathematically derived RX gate with Qiskit's
     built-in RX implementation for correctness checking.

Because RX(θ) is a very common gate in quantum computing, validating analytic
vs Qiskit behavior is both instructive and useful in debugging.

In [ ]:
def rx_matrix_numpy(theta: float) -> np.ndarray:
    """Analytic RX matrix using numpy: cos(theta/2) I - i sin(theta/2) X"""
    c = math.cos(theta / 2.0)
    s = math.sin(theta / 2.0)
    return c * np.eye(2, dtype=complex) - 1j * s * X

# Qiskit-Based RX Matrix

This function generates the RX(θ) gate using Qiskit’s native implementation.

Steps:
- Initialize a 1-qubit QuantumCircuit.
- Apply qc.rx(θ) to qubit 0.
- Convert the entire circuit into a matrix using Operator(qc).data.

Purpose:
- Verifies that our analytic matrix matches Qiskit's hardware-oriented
implementation.
- Ensures absolute confidence that the RX gate used in real devices obeys
the same mathematics as our numpy version.

This is a correctness and consistency check.

In [ ]:
def rx_matrix_qiskit(theta: float) -> np.ndarray:
    """Build a 1-qubit circuit with qiskit RX and return its Operator matrix."""
    qc = QuantumCircuit(1)
    qc.rx(theta, 0)
    return Operator(qc).data

# Global-Phase-Invariant Unitary Distance

Two unitaries U and V represent the same physical quantum operation
when:

    U = e^{iφ} V

because global phase e^{iφ} has no observable effect.

We use the distance metric:

    d(U,V) = 1 – |Tr(U V†)| / dim

Properties:
- d(U,V) = 0  ⟺  U = e^{iφ} V
- It is fast to compute
- It is mathematically robust
- It works even if two matrices only differ by floating-point roundoff

This function allows us to numerically check when RX(θ) ≈ X.

In [ ]:
def global_phase_invariant_distance(U: np.ndarray, V: np.ndarray) -> float:
    """
    Distance between unitaries up to global phase:
      d(U,V) = 1 - |Tr(U V†)|/dim
    This is 0 when U = e^{i phi} V.
    """
    dim = U.shape[0]
    tr = np.trace(U @ V.conj().T)
    return 1.0 - abs(tr) / dim

# Grid + Refinement Search for Best Theta

Although we know analytically that:

    θ = π + 2πk

this function demonstrates how one can *numerically discover* the value.

Why demonstrate numeric search?
- Useful in practice when analytic formulas are not obvious
- Mirrors calibration routines used in real quantum devices
- Shows how to match target gates through optimization methods

Process:
  1. Coarse grid search over [0, 2π)

     Evaluate d(RX(θ), X) for each θ.
     
     Keep the θ with minimal distance.

  2. Refinement
     Take a small window (e.g., ±0.2 rad) around the best coarse value.
     Evaluate many θ values in that window for high precision.

  3. Normalize θ back into [0, 2π).

Output:
- best_theta (radians)
- best_distance  (how close RX is to X)

In [ ]:
def find_theta_gridrefine(num_grid=2001, refine_radius=0.2, refine_steps=501):
    """
    Coarse-to-fine search for theta in [0, 2*pi).
    Returns best theta (in radians) found.
    """
    thetas = np.linspace(0.0, 2 * math.pi, num_grid, endpoint=False)
    best_idx = None
    best_val = 1.0
    for i, th in enumerate(thetas):
        U = rx_matrix_numpy(th)
        val = global_phase_invariant_distance(U, X)
        if val < best_val:
            best_val = val
            best_idx = i
    # refine around best theta
    center = thetas[best_idx]
    low = center - refine_radius
    high = center + refine_radius
    thetas_ref = np.linspace(low, high, refine_steps)
    best_theta = None
    best_val = 1.0
    for th in thetas_ref:
        # map th into [0,2pi)
        thm = ((th % (2 * math.pi)) + 2 * math.pi) % (2 * math.pi)
        U = rx_matrix_numpy(thm)
        val = global_phase_invariant_distance(U, X)
        if val < best_val:
            best_val = val
            best_theta = thm
    return best_theta, best_val

# Validation Function: Action-Level + Unitary-Level Checks

This function verifies that RX(θ) matches X up to global phase in two ways:

1. **Unitary-Level Check**
   Using global-phase-invariant distance, compare:
      - RX_numpy(θ) vs X
      - RX_qiskit(θ) vs X
      
   to ensure consistency between analytic and Qiskit implementations.

2. **State-Action Check**
   Compare the action on |0⟩

        RX(θ)|0⟩
        vs
        e^{iφ} X|0⟩
    
   If they differ only by global phase, the gate is valid.

The state-action check extracts the relative phase between two vectors
and computes the norm of the difference.

Return:
- distances
- global phase
- boolean validity flag

In [ ]:
def validate_theta(theta: float, tol=1e-9):
    """
    Validate that RX(theta) equals X up to global phase.
    Returns dict with distances and a Qiskit-verified matrix equality.
    """
    U_np = rx_matrix_numpy(theta)
    U_q = rx_matrix_qiskit(theta)
    d_np = global_phase_invariant_distance(U_np, X)
    d_q = global_phase_invariant_distance(U_q, X)
    # check action on basis states (up to phase)
    e0 = np.array([1.0, 0.0], dtype=complex)
    e1 = np.array([0.0, 1.0], dtype=complex)
    out_rx_on_e0 = U_np @ e0
    out_x_on_e0 = X @ e0
    # compute relative phase between vectors (if nonzero)
    # find scalar s such that out_rx_on_e0 ≈ s * out_x_on_e0
    if np.linalg.norm(out_x_on_e0) > 0:
        s = (out_rx_on_e0[0] / out_x_on_e0[0]) if out_x_on_e0[0] != 0 else (out_rx_on_e0[1] / out_x_on_e0[1])
    else:
        s = 0
    action_error = np.linalg.norm(out_rx_on_e0 - s * out_x_on_e0)
    return {
        "theta": theta,
        "distance_unitary_np": float(d_np),
        "distance_unitary_qiskit": float(d_q),
        "action_error_on_|0>": float(action_error),
        "global_phase_scalar_on_|0>": complex(s),
        "is_valid": (d_np < tol and action_error < 1e-8)
    }

# Main Routine: Numeric Search → Analytic Correction → Validation

This block controls the execution when the script is run directly.

Steps:

1. Print problem label:

        RX(θ) = X  (up to global phase)

2. Execute the numeric grid + refine search.
   This finds a θ very close to π (or 3π, etc).

3. Snap the numeric result to the nearest analytic solution:

        θ = π + 2πk

   Because numeric search introduces floating-point error, this correction
   ensures perfect symbolic correctness.

4. Validate the analytic θ using:
      - unitary comparison
      - action-level comparison

5. Print the final conclusion.

This mirrors a realistic workflow:
- discover parameter numerically
- infer analytic structure
- validate with exact math
- validate with Qiskit’s implementation

In [ ]:
if __name__ == "__main__":
    label = LatexNodes2Text().latex_to_text(r"RX(\theta) = X")
    print(f"Problem: {label}")

    # coarse numeric search (keeps previous behaviour)
    theta_found, val = find_theta_gridrefine()
    print(f"Numeric best theta (rad) = {theta_found:.12f}, numeric distance = {val:.3e}")

    # Snap to nearest analytical solution theta = pi + 2*pi*k
    k = int(round((theta_found - math.pi) / (2 * math.pi)))
    theta_analytic = math.pi + 2 * math.pi * k
    theta_analytic_norm = theta_analytic % (2 * math.pi)
    print("Analytical family: theta = pi + 2*pi*k")
    print(f"Nearest analytical theta (principal 0..2pi): {theta_analytic_norm:.12f}")

    # Validate using the analytic value to avoid small numeric offset issues
    result = validate_theta(theta_analytic)
    print(f"Validation (analytic theta): distance_unitary_np = {result['distance_unitary_np']:.3e}, "
          f"distance_unitary_qiskit = {result['distance_unitary_qiskit']:.3e}")
    print(f"Action on |0>: error = {result['action_error_on_|0>']:.3e}, "
          f"global-phase ≈ {result['global_phase_scalar_on_|0>']:.3f}")
    print(f"Conclusion: valid = {result['is_valid']}")

Problem: RX(θ) = X
Numeric best theta (rad) = 3.141622642269, numeric distance = 1.124e-10
Analytical family: theta = pi + 2*pi*k
Nearest analytical theta (principal 0..2pi): 3.141592653590
Validation (analytic theta): distance_unitary_np = 0.000e+00, distance_unitary_qiskit = 0.000e+00
Action on |0>: error = 6.123e-17, global-phase ≈ 0.000-1.000j
Conclusion: valid = True


# Results
Executing the program will result in the validation of the analytic θ (e.g., π) where RX(θ) implements the Pauli-X gate up to global phase, with unitary distances, action errors, and global phase scalars confirming equivalence.